In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str):
    """Calculate a mathematical expression."""
    return str(eval(expression))

@tool
def search_web(query: str):
    """Search the web for information."""
    return f"Found information about {query}"

tools = [calculator, search_web]
model_with_tools = model.bind_tools(tools)

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition


def llm_node(state):
    response = model_with_tools.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(MessagesState)

builder.add_node("llm", llm_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "llm")

builder.add_conditional_edges(
    "llm",
    tools_condition,
    {
        "tools": "tools",
        "__end__": END
    }
)

builder.add_edge("tools", "llm")

graph = builder.compile()